In [18]:
import xarray as xr
import matplotlib.pyplot as plt
from src.shash_torch import Shash
import numpy as np
import torch
from sklearn.model_selection import KFold
from src.shash_torch import Shash
from src.loss import ShashNLL  # wherever your class lives


In [36]:
model_spec = 'cnn3d_gelu_0_5/level=slgt_small_glade_new/opt=Adam_lr=0.001_batch=8_crit=ShashNLL'
#model_spec = 'cnn3d_3_layer/level=small_glade_new/opt=Adam_lr=0.001_batch=8_crit=ShashNLL'

In [37]:
results = xr.open_dataset(f'results/predictions/{model_spec}/predictions.nc')

In [38]:
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=False)

In [39]:
def fit_shash_and_eval(train_truth: xr.DataArray,
                       val_truth: xr.DataArray,
                       steps=2000,
                       lr=1e-2):

    x_train = torch.tensor(train_truth.values, dtype=torch.float32)
    x_val   = torch.tensor(val_truth.values, dtype=torch.float32)

    B, K = x_train.shape

    # parameters: (K,4) -> mu, log_sigma, gamma, log_tau
    params = torch.nn.Parameter(torch.zeros(K,4))

    # initialize
    params.data[:,0] = x_train.mean(dim=0)          # mu
    params.data[:,1] = torch.log(x_train.std(dim=0)+1e-3)  # log_sigma
    params.data[:,2] = 0.0                          # gamma
    params.data[:,3] = 0.0                          # log_tau

    optimizer = torch.optim.Adam([params], lr=lr)
    loss_fn = ShashNLL(reduction="mean")

    for _ in range(steps):
        optimizer.zero_grad()

        outputs = params.unsqueeze(0).expand(B,-1,-1).reshape(B,4*K)
        loss = loss_fn(outputs, x_train)
        #print(loss.item())

        loss.backward()
        optimizer.step()

    # validation NLL
    with torch.no_grad():
        Bv = x_val.shape[0]
        outputs = params.unsqueeze(0).expand(Bv,-1,-1).reshape(Bv,4*K)

        loss_fn_none = ShashNLL(reduction="none")
        val_losses = loss_fn_none(outputs, x_val)  # (Bv,K)

        nll_per_target = val_losses.mean(dim=0)  # (K,)

    return nll_per_target

In [40]:
losses = []
for fold, (train_idx, val_idx) in enumerate(kf.split(results['time'])):
    train_days = results['time'][train_idx]
    val_days = results['time'][val_idx]
    train_ds = results.sel(time = train_days)
    val_ds = results.sel(time = val_days)
    losses.append(fit_shash_and_eval(train_ds['truth'], val_ds['truth']))

In [41]:
losses = torch.stack(losses)
losses = losses.mean(dim=0)

In [42]:
losses

tensor([1.3096, 1.3813, 1.3845, 1.3758, 1.3646, 1.3463, 1.1283, 1.0743, 0.9984])